# Parte 4 â€” PINN: EstimaciÃ³n de parÃ¡metros Saint-Venant 1D

**Physics-Informed Neural Network** que aprende A(x,t) y Q(x,t) como funciones
continuas y estima los parÃ¡metros fÃ­sicos n y B_w minimizando:

- **L_data** = MSE entre Q predicho en x=0 y x=L vs observaciones del CSV
  *(ancla la soluciÃ³n a los datos reales en los bordes)*
- **L_pde** = residuos de las ecuaciones de Saint-Venant calculados por `torch.autograd`
  en 2 000 puntos de colocaciÃ³n interiores
  *(hace que la red respete la fÃ­sica en todo el dominio, no solo en los bordes)*

**Por quÃ© se necesitan los dos tÃ©rminos:** con solo L_data, el problema es no
identificable â€” infinitas combinaciones de (n, B_w, A, Q) ajustan los bordes.
L_pde resuelve la identifiabilidad porque n aparece en Sf y B_w en la geometrÃ­a
hidrÃ¡ulica: ambos deben hacer que los residuos SV sean â‰ˆ 0 en el interior.

**Entrenamiento:** Fase 1 Adam (exploraciÃ³n amplia) â†’ Fase 2 L-BFGS (convergencia fina).


In [ ]:
# ── Configuración ─────────────────────────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import DEFAULT_NUMERICAL, DEFAULT_PHYSICAL, DEFAULT_PINN

# Cambiar True/False para estimar otros parámetros
ESTIMATE_PARAMS = {"n": True, "Bw": True}

# Pesos de la función de pérdida
LAMBDA_DATA = DEFAULT_PINN.lambda_data
LAMBDA_PDE = DEFAULT_PINN.lambda_pde

# Hiperparámetros de entrenamiento
N_EPOCHS_ADAM = DEFAULT_PINN.n_epochs_adam
N_EPOCHS_WARMUP = DEFAULT_PINN.n_epochs_warmup
N_EPOCHS_RAMP = DEFAULT_PINN.n_epochs_ramp
N_ITER_LBFGS = DEFAULT_PINN.n_iter_lbfgs
N_COLLOC = DEFAULT_PINN.n_colloc
RESAMPLE_EVERY = DEFAULT_PINN.resample_every

# Conjeturas iniciales (lejos del valor verdadero para mostrar convergencia)
N_INIT = DEFAULT_PINN.n_init
BW_INIT = DEFAULT_PINN.bw_init

# Archivo de observaciones (mismo que en calibración, notebook 03)
CSV_NAME = "series_corta_shift.csv"
WARMUP_SEC = DEFAULT_NUMERICAL.warmup_seconds

# Nuevos términos de pérdida — tercera ronda PINN
LAMBDA_H        = DEFAULT_PINN.lambda_h        # 1.0  — peso L_h (profundidad)
LAMBDA_STEADY   = DEFAULT_PINN.lambda_steady    # 0.1  — peso L_steady (t=0)
N_COLLOC_STEADY = DEFAULT_PINN.n_colloc_steady  # 200  — puntos en t=0

In [ ]:
# ── Carga de datos ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
import importlib, importlib.util, sys
from pathlib import Path
import numpy as np, pandas as pd, torch

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# Parámetros verdaderos desde sinteticos.py
_spec = importlib.util.spec_from_file_location("sinteticos", ROOT / "data" / "sinteticos.py")
sinteticos = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(sinteticos)
TRUE_N, TRUE_BW = sinteticos.N_MANN, sinteticos.B_W

csv_path = ROOT / "data" / "synthetic" / CSV_NAME
df   = pd.read_csv(csv_path, parse_dates=["datetime"])
t_sec = (df["datetime"] - df["datetime"].iloc[0]).dt.total_seconds().to_numpy(dtype=float)
q_up  = df["Q_upstream_m3s"].to_numpy(dtype=float)
q_dn  = df["Q_downstream_m3s"].to_numpy(dtype=float)

t_tensor  = torch.tensor(t_sec, dtype=torch.float32)
qu_tensor = torch.tensor(q_up,  dtype=torch.float32)
qd_tensor = torch.tensor(q_dn,  dtype=torch.float32)
h_outlet = df["h_outlet_m"].to_numpy(dtype=float)
hL_tensor = torch.tensor(h_outlet, dtype=torch.float32)
print(f"h_outlet: min={h_outlet.min():.3f} m  max={h_outlet.max():.3f} m  media={h_outlet.mean():.3f} m")
T_total   = float(t_sec[-1])

print(f"CSV: {CSV_NAME}  |  pasos: {len(df)}  |  T={T_total/3600:.1f} h")
print(f"Valores verdaderos: n={TRUE_N}  B_w={TRUE_BW} m")

In [ ]:
# ── Construcción y entrenamiento ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
import importlib
import src.pinn as pinn_mod
importlib.reload(pinn_mod)

model = pinn_mod.SVPINN(
    hidden_size=DEFAULT_PINN.hidden_size,
    n_layers=DEFAULT_PINN.n_layers,
    S0=DEFAULT_PHYSICAL.bed_slope,
    beta=DEFAULT_PINN.beta,
    n_init=N_INIT,
    Bw_init=BW_INIT,
    estimate_params=ESTIMATE_PARAMS,
)

# Curriculum 3 fases:
#   Fase 1a: épocas 0 – N_EPOCHS_WARMUP         → solo L_data (sin física)
#   Fase 1b: épocas WARMUP – WARMUP+RAMP         → ramp lineal lambda_pde 0→0.05
#   Fase 1c: épocas WARMUP+RAMP – N_EPOCHS_ADAM  → L_data + L_pde completa
#   Fase 2:  L-BFGS convergencia fina
result = pinn_mod.train(
    model,
    x0_data=qu_tensor,
    xL_data=qd_tensor,
    hL_data=hL_tensor,
    t_data=t_tensor,
    L=DEFAULT_PHYSICAL.channel_length,
    T=T_total,
    lambda_data=LAMBDA_DATA,
    lambda_h=LAMBDA_H,
    lambda_pde=LAMBDA_PDE,
    lambda_steady=LAMBDA_STEADY,
    n_colloc_steady=N_COLLOC_STEADY,
    n_epochs_adam=N_EPOCHS_ADAM,
    n_epochs_warmup=N_EPOCHS_WARMUP,
    n_epochs_ramp=N_EPOCHS_RAMP,
    n_iter_lbfgs=N_ITER_LBFGS,
    n_colloc=N_COLLOC,
    resample_every=RESAMPLE_EVERY,
    t_warmup=WARMUP_SEC,
    verbose_every=500,
)
print(f"\nEstimado PINN: n={result.n_estimate:.5f}  Bw={result.Bw_estimate:.3f} m")

In [ ]:
# ── Curvas de pérdida + estimación de parámetros ──────────────────────────────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
from IPython.display import display

history = result.loss_history
adam_entries = [h for h in history if "data" in h]
epochs_a = [h["epoch"] for h in adam_entries]
totals_a = [h["total"] for h in adam_entries]
datas_a  = [h["data"]  for h in adam_entries]
pdes_a   = [h["pde"]   for h in adam_entries]
hs_a      = [h["h"]      for h in adam_entries]
steadys_a = [h["steady"] for h in adam_entries]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Curvas de pérdida (fase Adam, eje log)
ax = axes[0]
ax.semilogy(epochs_a, totals_a, label="L_total", color="k")
ax.semilogy(epochs_a, datas_a,  label="L_data",  color="steelblue")
ax.semilogy(epochs_a, pdes_a,   label=f"L_pde (x{LAMBDA_PDE})", color="coral")
ax.semilogy(epochs_a, hs_a,      label="L_h",      color="seagreen")
ax.semilogy(epochs_a, steadys_a, label="L_steady",  color="mediumpurple")
ax.set_xlabel("Época (Adam)")
ax.set_ylabel("Pérdida (escala log)")
ax.set_title("Curvas de pérdida — fase Adam")
ax.legend(); ax.grid(True, alpha=0.3)

# Tabla y gráfico de parámetros
glue_path = ROOT / "data" / "synthetic" / "glue_parametros_aceptados.csv"
table = pinn_mod.build_comparison_table(
    n_pinn=result.n_estimate, Bw_pinn=result.Bw_estimate,
    n_true=TRUE_N, Bw_true=TRUE_BW,
    glue_csv_path=str(glue_path),
)
display(table)

params     = ["n (Manning)", "B_w [m]"]
verdaderos = [TRUE_N, TRUE_BW]
pinns      = [result.n_estimate, result.Bw_estimate]
use_glue   = "calibracion_GLUE" in table.columns
calibs     = list(table["calibracion_GLUE"]) if use_glue else [np.nan, np.nan]

ax2 = axes[1]
x = np.arange(len(params)); w = 0.25
ax2.bar(x - w, verdaderos, w, label="Verdadero",         color="0.70")
ax2.bar(x,     pinns,      w, label="PINN",               color="steelblue")
if use_glue and not any(np.isnan(np.array(calibs, dtype=float))):
    ax2.bar(x + w, calibs, w, label="Calibración (GLUE)", color="coral")
ax2.set_xticks(x); ax2.set_xticklabels(params)
ax2.set_title("Parámetros estimados")
ax2.legend(); ax2.grid(True, axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(ROOT / "figures" / "pinn_parametros.png", dpi=150)
plt.show()

In [ ]:
# â”€â”€ Hidrograma: PINN vs observado en x=L â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
model.eval()
with torch.no_grad():
    t_norm_all = t_tensor / T_total
    _, Q_pinn  = model(torch.ones_like(t_norm_all), t_norm_all)
Q_pinn_np = Q_pinn.numpy()

t_h       = t_sec / 3600.0
mask_post = t_sec >= WARMUP_SEC

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t_h[mask_post], q_dn[mask_post],    "r.",        ms=3,  alpha=0.6, label="Q_obs  (x=L)")
ax.plot(t_h[mask_post], Q_pinn_np[mask_post], "steelblue", lw=1.5, label="Q_PINN (x=L)")
ax.axvspan(0, WARMUP_SEC / 3600, color="gray", alpha=0.12, label="Warm-up")
ax.set_xlabel("Tiempo (h)")
ax.set_ylabel("Q (mÂ³/s)")
ax.set_title(
    f"Hidrograma salida â€” PINN vs observado\n"
    f"n={result.n_estimate:.4f} (verd. {TRUE_N})  "
    f"Bw={result.Bw_estimate:.2f} m (verd. {TRUE_BW} m)"
)
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(ROOT / "figures" / "pinn_hidrograma.png", dpi=150)
plt.show()
